# 04 — Feature Engineering (PySpark)

## Objective
Turn five cleaned tables into one modeling matrix: one row per policy,
quote-time features only, three targets.

## Targets (all saved, the choice is made in Notebook 05)
  claim_amount    pure premium, zero for ~63.5% of policies (Tweedie target)
  has_claim       frequency (two-part model, part 1)
  severity_ratio  claim_amount / IDV for claimants only (part 2)
                  Third-party claims are not capped by IDV, so this
                  ratio can exceed 1.0 for them.

## Leakage Rules (third project where this is the central risk)
1. Quote-time only: nothing that exists only after an accident
2. Premium columns are the BASELINE to beat, never inputs
3. Generator artifacts are excluded: vehicle_age_missing, ncb_missing
4. No target encoding in this notebook (fit inside CV folds in NB05)
5. A univariate leakage screen runs before anything is saved

## Output
  modeling_data.csv, feature_spec.json, baseline_premium.csv
feature_spec.json records every column and category level, so the
same matrix can be rebuilt for a new quote at inference time.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings("ignore")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_PATH   = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")
PROCESSED  = "../data/processed"
LOCAL_TEMP = os.path.abspath("../data/spark_temp")
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Insurance_FeatureEngineering")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def read_csv(name):
    return (spark.read.option("header", "true")
            .option("inferSchema", "true")
            .csv(f"{PROCESSED}/{name}"))

policyholders = read_csv("policyholders_clean.csv")
vehicles      = read_csv("vehicles_clean.csv")
policies      = read_csv("policies_clean.csv")
labels        = read_csv("labels_clean.csv")

# One row per policy. vehicles.idv is dropped: NB02 verified it equals
# policies.idv for every policy.
policy_base = (
    policies
    .join(policyholders,        on="policyholder_id", how="inner")
    .join(vehicles.drop("idv"), on="policyholder_id", how="inner")
    .join(labels,               on="policy_id",       how="inner")
)
policy_base.cache()

n_rows     = policy_base.count()
n_distinct = policy_base.select("policy_id").distinct().count()
rate       = policy_base.agg(F.avg("has_claim")).collect()[0][0] * 100

print("Data loaded and joined ✅")
print(f"  rows            : {n_rows:,}   (NB02 final: 97,978)")
print(f"  distinct ids    : {n_distinct:,}   (must equal rows)")
print(f"  claim rate      : {rate:.4f}%   (NB02 final: 36.4868%)")
print(f"  columns         : {len(policy_base.columns)}")

Data loaded and joined ✅
  rows            : 97,978   (NB02 final: 97,978)
  distinct ids    : 97,978   (must equal rows)
  claim rate      : 36.4868%   (NB02 final: 36.4868%)
  columns         : 47


In [2]:
print("STEP 1 — BUILD FEATURES\n")

features = (
    policy_base

    # ── Driver ────────────────────────────────────────────────
    .withColumn("log_income", F.log1p("monthly_income"))
    .withColumn("is_young_driver",  (F.col("age") < 25).cast("int"))
    .withColumn("is_senior_driver", (F.col("age") >= 60).cast("int"))
    # Share of adult life spent driving. Clipped to 1.0 because the
    # ages repaired in NB02 can sit below the experience that was
    # generated for the original (impossible) age.
    .withColumn(
        "experience_ratio",
        F.round(F.least(
            F.lit(1.0),
            F.col("driving_exp_years") /
            F.greatest(F.col("age") - 17, F.lit(1.0))
        ), 4)
    )
    .withColumn("has_prior_claim", (F.col("prior_claims_count") > 0).cast("int"))

    # ── Vehicle ───────────────────────────────────────────────
    .withColumn("log_idv", F.log("idv"))
    .withColumn("is_new_vehicle", (F.col("vehicle_age_years") <= 1).cast("int"))
    .withColumn("is_old_vehicle", (F.col("vehicle_age_years") >= 8).cast("int"))
    .withColumn("idv_to_annual_income",
                F.round(F.col("idv") / (F.col("monthly_income") * 12), 4))

    # ── Policy ────────────────────────────────────────────────
    .withColumn("has_od_cover",
                (F.col("policy_type") != "third_party_only").cast("int"))
    .withColumn("has_deductible", (F.col("voluntary_deductible") > 0).cast("int"))

    # ── Targets ───────────────────────────────────────────────
    # Defined only for claimants; null otherwise (a zero would be a lie,
    # since severity of "no claim" does not exist).
    .withColumn("severity_ratio",
                F.when(F.col("has_claim") == 1,
                       F.col("claim_amount") / F.col("idv")))
)

features.createOrReplaceTempView("features")

new_cols = ["log_income", "experience_ratio", "log_idv",
            "idv_to_annual_income", "severity_ratio"]
print("New numeric features:")
features.select(*new_cols).summary("min", "25%", "50%", "75%", "max").show()

print("New binary features (share of policies = 1):")
bin_cols = ["is_young_driver", "is_senior_driver", "has_prior_claim",
            "is_new_vehicle", "is_old_vehicle", "has_od_cover", "has_deductible"]
features.select([F.round(F.avg(c) * 100, 1).alias(c) for c in bin_cols]).show()

STEP 1 — BUILD FEATURES

New numeric features:
+-------+------------------+----------------+------------------+--------------------+--------------------+
|summary|        log_income|experience_ratio|           log_idv|idv_to_annual_income|      severity_ratio|
+-------+------------------+----------------+------------------+--------------------+--------------------+
|    min| 9.268584751497448|          0.0192|10.239959789157341|              0.0019|2.864716375312165...|
|    25%|12.128478441989028|          0.2854| 12.65714826489039|              0.0624|0.028533973866256723|
|    50%|12.766564649476203|             0.5|13.058358047428417|               0.127|  0.0841472340425532|
|    75%|13.302275951594922|          0.7239| 13.53579665516167|              0.2941| 0.29495439567430026|
|    max|14.038645397240535|             1.0|15.706662771070365|             33.1267|   1.976529129872529|
+-------+------------------+----------------+------------------+--------------------+------------

In [3]:
print("STEP 2 — SHOULD THE NB02 MISSING-VALUE FLAGS BE FEATURES?\n")

for flag in ["vehicle_age_missing", "ncb_missing", "premium_was_missing"]:
    print(f"── {flag} ──")
    spark.sql(f"""
        SELECT {flag}                                 AS flag,
               COUNT(*)                               AS policies,
               ROUND(AVG(has_claim) * 100, 2)         AS claim_rate_pct,
               ROUND(AVG(claim_amount), 0)            AS pure_premium
        FROM features
        GROUP BY 1
        ORDER BY 1
    """).show()

print("""DECISION: all three flags are excluded.

  premium_was_missing : premium is not a feature, so its flag is moot.

  vehicle_age_missing : the claim-rate gap in NB02 (+7.8 pp) matches the
      8+ years-old bucket. My reading of the generator is that a missing
      age becomes NaN, every age comparison fails, and the code falls
      through to the old-vehicle branch. That is a hypothesis from the
      code, not something I tested.

  ncb_missing : same mechanism, with a smaller effect.

  If the hypothesis is right, a model that learns these flags is
  learning a simulation bug. On REAL data a missingness flag is often
  among the most valuable features (has_credit_history in the loan
  project was one). So this is a simulation-specific call, not a rule.
""")

STEP 2 — SHOULD THE NB02 MISSING-VALUE FLAGS BE FEATURES?

── vehicle_age_missing ──
+----+--------+--------------+------------+
|flag|policies|claim_rate_pct|pure_premium|
+----+--------+--------------+------------+
|   0|   95999|         36.33|     78520.0|
|   1|    1979|         44.16|    141407.0|
+----+--------+--------------+------------+

── ncb_missing ──
+----+--------+--------------+------------+
|flag|policies|claim_rate_pct|pure_premium|
+----+--------+--------------+------------+
|   0|   96013|         36.44|     79945.0|
|   1|    1965|         38.73|     72231.0|
+----+--------+--------------+------------+

── premium_was_missing ──
+----+--------+--------------+------------+
|flag|policies|claim_rate_pct|pure_premium|
+----+--------+--------------+------------+
|   0|   96550|         36.47|     79952.0|
|   1|    1428|         37.61|     68853.0|
+----+--------+--------------+------------+

DECISION: all three flags are excluded.

  premium_was_missing : premium is 

In [4]:
print("STEP 3 — ASSEMBLE THE MODELING MATRIX\n")

CARRY   = ["policy_id", "idv"]     # identifiers / conversion helpers, not features
TARGETS = ["claim_amount", "has_claim", "severity_ratio"]

NUMERIC = [
    # driver
    "age", "driving_exp_years", "log_income", "prior_claims_count",
    "ncb_percentage", "is_young_driver", "is_senior_driver",
    "experience_ratio", "has_prior_claim",
    # vehicle
    "vehicle_age_years", "engine_cc", "log_idv", "is_modified",
    "is_new_vehicle", "is_old_vehicle", "idv_to_annual_income",
    # policy
    "voluntary_deductible", "has_deductible", "addon_count",
    "has_zero_depreciation", "has_engine_protection",
    "has_roadside_assistance", "has_consumables", "has_od_cover",
    "policy_start_quarter",
]
CATEGORICAL = ["vehicle_make", "vehicle_segment", "vehicle_type", "fuel_type",
               "policy_type", "city_tier", "occupation", "gender"]

pdf = features.select(CARRY + TARGETS + NUMERIC + CATEGORICAL).toPandas()

# Fix the category levels explicitly, so a new quote with a missing
# level still produces the same columns at inference time
LEVELS = {c: sorted(pdf[c].dropna().astype(str).unique()) for c in CATEGORICAL}
for c in CATEGORICAL:
    pdf[c] = pd.Categorical(pdf[c].astype(str), categories=LEVELS[c])

dummies = pd.get_dummies(pdf[CATEGORICAL], prefix=CATEGORICAL, dtype=int)
model   = pd.concat([pdf[CARRY + TARGETS + NUMERIC], dummies], axis=1)

FEATURE_COLS = NUMERIC + list(dummies.columns)

print(f"Rows                : {len(model):,}")
print(f"Numeric features    : {len(NUMERIC)}")
print(f"One-hot features    : {dummies.shape[1]} "
      f"(from {len(CATEGORICAL)} categorical columns)")
print(f"Total features      : {len(FEATURE_COLS)}")

print("\nCategory levels:")
for c, lv in LEVELS.items():
    print(f"  {c:<16} {len(lv):>2}  {lv}")

nulls = model[FEATURE_COLS].isnull().sum()
print(f"\nFeature columns with nulls : {int((nulls > 0).sum())}  (must be 0)")
if (nulls > 0).any():
    print(nulls[nulls > 0])

print(f"\nTarget summary:")
print(f"  has_claim mean              : {model['has_claim'].mean()*100:.2f}%")
print(f"  claim_amount zero share     : {(model['claim_amount']==0).mean()*100:.2f}%")
print(f"  severity_ratio non-null     : {model['severity_ratio'].notna().sum():,} "
      f"(must equal number of claimants)")

STEP 3 — ASSEMBLE THE MODELING MATRIX

Rows                : 97,978
Numeric features    : 25
One-hot features    : 41 (from 8 categorical columns)
Total features      : 66

Category levels:
  vehicle_make     12  ['Audi', 'BMW', 'Bajaj', 'Honda', 'Hyundai', 'Kia', 'MG', 'Mahindra', 'Maruti Suzuki', 'Mercedes-Benz', 'Tata', 'Toyota']
  vehicle_segment   5  ['economy', 'luxury', 'mid', 'premium', 'two_wheeler']
  vehicle_type      4  ['commercial_vehicle', 'private_car', 'taxi', 'two_wheeler']
  fuel_type         4  ['cng', 'diesel', 'electric', 'petrol']
  policy_type       3  ['comprehensive', 'standalone_od', 'third_party_only']
  city_tier         4  ['Metro', 'Tier1', 'Tier2', 'Tier3']
  occupation        5  ['business', 'retired', 'salaried', 'self_employed', 'student']
  gender            4  ['Female', 'Male', 'Not Specified', 'Other']

Feature columns with nulls : 0  (must be 0)

Target summary:
  has_claim mean              : 36.49%
  claim_amount zero share     : 63.51%
  sever

In [5]:
from sklearn.metrics import roc_auc_score

print("STEP 4 — LEAKAGE GUARD\n")

FORBIDDEN_EXACT = {
    "has_claim", "severity_ratio", "claim_amount",
    "total_premium", "base_od_premium", "tp_premium", "addon_premium",
    "ncb_discount", "deductible_discount", "premium_was_missing",
    "vehicle_age_missing", "ncb_missing",
    "is_fraud_suspected", "monsoon_claim", "was_capped_at_idv",
    "settlement_days", "surveyor_assessment", "exposure_years",
    "insurer",
}
FORBIDDEN_PREFIX = ("claim_",)   # claim_type, claim_month, claim_prob, ...

bad = [c for c in FEATURE_COLS
       if c in FORBIDDEN_EXACT or c.startswith(FORBIDDEN_PREFIX)]
assert not bad, f"Forbidden columns in feature list: {bad}"
print("✅ No forbidden column is in the feature list")
print(f"   idv is carried but is NOT a feature (log_idv is): "
      f"{'idv' not in FEATURE_COLS}")

# ── Univariate screen ─────────────────────────────────────────
print("\nUNIVARIATE SCREEN — each feature alone, against two targets\n")
y   = model["has_claim"].values
amt = model["claim_amount"]

rows = []
constant = []
for c in FEATURE_COLS:
    x = model[c].values.astype(float)
    if np.nanstd(x) == 0:
        constant.append(c)
        continue
    auc = roc_auc_score(y, x)
    rho = pd.Series(x).corr(amt, method="spearman")
    rows.append((c, max(auc, 1 - auc), abs(rho)))

screen = (pd.DataFrame(rows, columns=["feature", "freq_auc", "abs_rho_pure"])
          .sort_values("freq_auc", ascending=False).reset_index(drop=True))

print(f"{'Feature':<30} {'Freq AUC':>9} {'|Spearman| pure':>16}")
for _, r in screen.head(12).iterrows():
    flag = "  <- INVESTIGATE" if (r["freq_auc"] > 0.70 or r["abs_rho_pure"] > 0.60) else ""
    print(f"  {r['feature']:<28} {r['freq_auc']:>9.4f} {r['abs_rho_pure']:>16.4f}{flag}")

print(f"\nConstant features (zero variance): {constant if constant else 'none'}")
suspicious = screen[(screen.freq_auc > 0.70) | (screen.abs_rho_pure > 0.60)]
print(f"Features over threshold : {len(suspicious)}")
print("""
How to read this:
  The best single factors in NB01 moved claim rate between roughly
  24% and 44%, so no honest single feature should reach a frequency AUC
  near 0.8. Anything above 0.70 means a feature may be encoding the
  label. In the fraud, churn and loan projects, the perfect scores
  started as a feature like this.
  A high Spearman against claim_amount is expected for log_idv, because
  severity scales with the vehicle's value. That is a legitimate driver,
  not leakage. Judge it against the frequency AUC.
""")

STEP 4 — LEAKAGE GUARD

✅ No forbidden column is in the feature list
   idv is carried but is NOT a feature (log_idv is): True

UNIVARIATE SCREEN — each feature alone, against two targets

Feature                         Freq AUC  |Spearman| pure
  prior_claims_count              0.6003           0.1804
  has_prior_claim                 0.5885           0.1654
  ncb_percentage                  0.5874           0.1517
  engine_cc                       0.5466           0.0977
  vehicle_type_two_wheeler        0.5462           0.1034
  vehicle_age_years               0.5451           0.0566
  city_tier_Metro                 0.5405           0.0910
  log_income                      0.5310           0.0545
  vehicle_type_taxi               0.5304           0.1316
  vehicle_type_commercial_vehicle    0.5302           0.0966
  is_old_vehicle                  0.5283           0.0530
  city_tier_Tier2                 0.5266           0.0605

Constant features (zero variance): none
Features over

In [6]:
print("STEP 5 — SAVE\n")

model[CARRY + TARGETS + FEATURE_COLS].to_csv(
    f"{PROCESSED}/modeling_data.csv", index=False)
print(f"  ✅ modeling_data.csv   {model.shape}")

# Baseline premium and policy id only. The premium is never a feature.
features.select("policy_id", "total_premium").toPandas().to_csv(
    f"{PROCESSED}/baseline_premium.csv", index=False)
print("  ✅ baseline_premium.csv  (evaluation only)")

spec = {
    "feature_cols"   : FEATURE_COLS,
    "numeric"        : NUMERIC,
    "categorical"    : CATEGORICAL,
    "category_levels": LEVELS,
    "targets"        : TARGETS,
    "carry"          : CARRY,
    "forbidden"      : sorted(FORBIDDEN_EXACT) + ["claim_*"],
}
with open(f"{PROCESSED}/feature_spec.json", "w") as f:
    json.dump(spec, f, indent=2)
print(f"  ✅ feature_spec.json    ({len(FEATURE_COLS)} features)")

oracle_ok = os.path.exists(f"{PROCESSED}/oracle_claim_prob.csv")
print(f"  oracle_claim_prob.csv present: {oracle_ok}  (untouched, evaluation only)")

# ── Severity target ───────────────────────────────────────────
sr = model["severity_ratio"].dropna()
print(f"\nseverity_ratio (claimants, n={len(sr):,}):")
print(f"  P10 {sr.quantile(.10):.4f} | median {sr.median():.4f} | "
      f"P90 {sr.quantile(.90):.4f} | P99 {sr.quantile(.99):.4f} | max {sr.max():.2f}")
print(f"  share exactly 1.0 (paid at IDV): {(sr >= 0.9999).mean()*100:.1f}%")
print(f"  share above 1.0 (third-party)  : {(sr > 1.0001).mean()*100:.1f}%")

print(f"""
SUMMARY
  Rows               : {len(model):,}
  Features           : {len(FEATURE_COLS)}  ({len(NUMERIC)} numeric + {dummies.shape[1]} one-hot)
  Claim rate         : {model['has_claim'].mean()*100:.2f}%
  Leakage guard      : passed
  Screen flags       : {len(suspicious)}

EXCLUDED ON PURPOSE
  Premium columns, insurer, exposure_years (constant 1.0)
  Missing-value flags (generator artifacts)
  All post-claim columns, and claim_prob (oracle)
""")

policy_base.unpersist()
print("Cache cleared ✅")

STEP 5 — SAVE

  ✅ modeling_data.csv   (97978, 71)
  ✅ baseline_premium.csv  (evaluation only)
  ✅ feature_spec.json    (66 features)
  oracle_claim_prob.csv present: True  (untouched, evaluation only)

severity_ratio (claimants, n=35,749):
  P10 0.0091 | median 0.0841 | P90 0.7202 | P99 1.0000 | max 1.98
  share exactly 1.0 (paid at IDV): 4.1%
  share above 1.0 (third-party)  : 0.5%

SUMMARY
  Rows               : 97,978
  Features           : 66  (25 numeric + 41 one-hot)
  Claim rate         : 36.49%
  Leakage guard      : passed
  Screen flags       : 0

EXCLUDED ON PURPOSE
  Premium columns, insurer, exposure_years (constant 1.0)
  Missing-value flags (generator artifacts)
  All post-claim columns, and claim_prob (oracle)

Cache cleared ✅
